# 03: SQL analysis with DuckDB

Reshape the match data to one row per team per match, check the logic with a
league table, and build rolling-form features using only past matches.

In [10]:
import duckdb

con = duckdb.connect()   # in-memory database, nothing to set up

con.sql("""
    CREATE VIEW matches AS
    SELECT * FROM read_csv_auto('../data/processed/matches.csv')
""")

con.sql("SELECT * FROM matches LIMIT 5").df()

,Season,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HST,...,AF,HC,AC,HY,AY,HR,AR,B365H,B365D,B365A
0,1718,2017-08-11,Arsenal,Leicester,4,3,H,27,6,10,...,12,9,4,0,1,0,0,1.53,4.5,6.50
1,1718,2017-08-12,Brighton,Man City,0,2,A,6,14,2,...,9,3,10,0,2,0,0,11.00,5.5,1.33
2,1718,2017-08-12,Chelsea,Burnley,2,3,A,19,10,6,...,11,8,5,3,3,2,0,1.25,6.5,15.00
3,1718,2017-08-12,Crystal Palace,Huddersfield,0,3,A,14,8,4,...,19,12,9,1,3,0,0,1.83,3.6,5.00
4,1718,2017-08-12,Everton,Stoke,1,0,H,9,9,4,...,10,6,7,1,1,0,0,1.70,3.8,5.75


1. Matches per season

In [11]:
con.sql("""
    SELECT Season, COUNT(*) AS n_matches
    FROM matches
    GROUP BY Season
    ORDER BY Season
""").df()


,Season,n_matches
0,1718,380
1,1819,380
2,1920,380
3,2021,380
4,2122,380
5,2223,380
6,2324,380
7,2425,380
8,2526,380
9,2627,100


2. Home win rate per season

In [12]:
con.sql("""
    SELECT Season,
           AVG(CASE WHEN FTR = 'H' THEN 1 ELSE 0 END) AS home_win_rate
    FROM matches
    GROUP BY Season
    ORDER BY Season
""").df()

,Season,home_win_rate
0,1718,0.455263
1,1819,0.476316
2,1920,0.452632
3,2021,0.378947
4,2122,0.428947
5,2223,0.484211
6,2324,0.460526
7,2425,0.407895
8,2526,0.426316
9,2627,0.420000


3. Average goals per match per season

In [13]:
con.sql("""
    SELECT Season,
           AVG(FTHG + FTAG) AS goals_per_match
    FROM matches
    GROUP BY Season
    ORDER BY Season
""").df()

,Season,goals_per_match
0,1718,2.678947
1,1819,2.821053
2,1920,2.721053
3,2021,2.694737
4,2122,2.818421
5,2223,2.852632
6,2324,3.278947
7,2425,2.934211
8,2526,2.750000
9,2627,2.730000


## Reshape the data (one row per team per match)

Each match becomes two rows, one from the home team's point of view and one
from the away team's. This makes league tables and rolling form much easier
to calculate, since a team's goals and points are in a single column.

In [14]:
con.sql("""
    CREATE OR REPLACE VIEW team_matches AS
    SELECT Season, Date, HomeTeam AS Team, AwayTeam AS Opponent, 'H' AS Venue,
           FTHG AS GoalsFor, FTAG AS GoalsAgainst,
           CASE FTR WHEN 'H' THEN 3 WHEN 'D' THEN 1 ELSE 0 END AS Points
    FROM matches
    UNION ALL
    SELECT Season, Date, AwayTeam, HomeTeam, 'A',
           FTAG, FTHG,
           CASE FTR WHEN 'A' THEN 3 WHEN 'D' THEN 1 ELSE 0 END
    FROM matches
""")

In [15]:
con.sql("""
    SELECT (SELECT COUNT(*) FROM team_matches) AS team_rows,
           (SELECT COUNT(*) FROM matches) * 2 AS expected
""").df()

,team_rows,expected
0,7040,7040


In [16]:
con.sql("""
    SELECT Season, Team, COUNT(*) AS played
    FROM team_matches
    GROUP BY Season, Team
    ORDER BY played, Season
    LIMIT 10
""").df()


,Season,Team,played
0,2627,Leeds,10
1,2627,Ipswich,10
2,2627,Hull,10
3,2627,Chelsea,10
4,2627,Aston Villa,10
5,2627,Man City,10
6,2627,Coventry,10
7,2627,Man United,10
8,2627,Newcastle,10
9,2627,Fulham,10


In [17]:
con.sql("""
    SELECT * FROM team_matches
    WHERE Team = 'Arsenal'
    ORDER BY Date
    LIMIT 10
""").df()

,Season,Date,Team,Opponent,Venue,GoalsFor,GoalsAgainst,Points
0,1718,2017-08-11,Arsenal,Leicester,H,4,3,3
1,1718,2017-08-19,Arsenal,Stoke,A,0,1,0
2,1718,2017-08-27,Arsenal,Liverpool,A,0,4,0
3,1718,2017-09-09,Arsenal,Bournemouth,H,3,0,3
4,1718,2017-09-17,Arsenal,Chelsea,A,0,0,1
5,1718,2017-09-25,Arsenal,West Brom,H,2,0,3
6,1718,2017-10-01,Arsenal,Brighton,H,2,0,3
7,1718,2017-10-14,Arsenal,Watford,A,1,2,0
8,1718,2017-10-22,Arsenal,Everton,A,5,2,3
9,1718,2017-10-28,Arsenal,Swansea,H,2,1,3


## League table check

In [18]:
 con.sql("""
    SELECT Team,
           COUNT(*) AS played,
           SUM(CASE WHEN Points = 3 THEN 1 ELSE 0 END) AS wins,
           SUM(CASE WHEN Points = 1 THEN 1 ELSE 0 END) AS draws,
           SUM(CASE WHEN Points = 0 THEN 1 ELSE 0 END) AS losses,
           SUM(GoalsFor) AS goals_for,
           SUM(GoalsAgainst) AS goals_against,
           SUM(GoalsFor) - SUM(GoalsAgainst) AS goal_diff,
           SUM(Points) AS points
    FROM team_matches
    WHERE Season = '2526'
    GROUP BY Team
    ORDER BY points DESC, goal_diff DESC
""").df()

,Team,played,wins,draws,losses,goals_for,goals_against,goal_diff,points
0,Arsenal,38,26.0,7.0,5.0,71.0,27.0,44.0,85.0
1,Man City,38,23.0,9.0,6.0,77.0,35.0,42.0,78.0
2,Man United,38,20.0,11.0,7.0,69.0,50.0,19.0,71.0
3,Aston Villa,38,19.0,8.0,11.0,56.0,49.0,7.0,65.0
4,Liverpool,38,17.0,9.0,12.0,63.0,53.0,10.0,60.0
5,Bournemouth,38,13.0,18.0,7.0,58.0,54.0,4.0,57.0
6,Sunderland,38,14.0,12.0,12.0,42.0,48.0,-6.0,54.0
7,Brighton,38,14.0,11.0,13.0,52.0,46.0,6.0,53.0
8,Brentford,38,14.0,11.0,13.0,55.0,52.0,3.0,53.0
9,Chelsea,38,14.0,10.0,14.0,58.0,52.0,6.0,52.0


In [19]:
con.sql("""
    SELECT SUM(GoalsFor) AS total_for, SUM(GoalsAgainst) AS total_against
    FROM team_matches WHERE Season = '2526'
""").df()

,total_for,total_against
0,1045.0,1045.0


## Rolling form features

In [20]:
con.sql("""
    CREATE OR REPLACE VIEW team_form AS
    SELECT Season, Date, Team, Opponent, Venue, GoalsFor, GoalsAgainst, Points,
           COUNT(*)           OVER w5  AS prior_matches,
           SUM(Points)        OVER w5  AS points_last5,
           AVG(GoalsFor)      OVER w5  AS goals_for_last5,
           AVG(GoalsAgainst)  OVER w5  AS goals_against_last5,
           AVG(GoalsFor)      OVER w10 AS goals_for_last10,
           AVG(GoalsAgainst)  OVER w10 AS goals_against_last10,
           date_diff('day',
                     LAG(Date) OVER (PARTITION BY Team ORDER BY Date),
                     Date)             AS days_rest
    FROM team_matches
    WINDOW
        w5  AS (PARTITION BY Team ORDER BY Date
                ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING),
        w10 AS (PARTITION BY Team ORDER BY Date
                ROWS BETWEEN 10 PRECEDING AND 1 PRECEDING)
""")

In [21]:
con.sql("""
    SELECT * FROM team_form
    WHERE Team = 'Arsenal'
    ORDER BY Date
    LIMIT 12
""").df()

,Season,Date,Team,Opponent,Venue,GoalsFor,GoalsAgainst,Points,prior_matches,points_last5,goals_for_last5,goals_against_last5,goals_for_last10,goals_against_last10,days_rest
0,1718,2017-08-11,Arsenal,Leicester,H,4,3,3,0,NaN,NaN,NaN,NaN,NaN,<NA>
1,1718,2017-08-19,Arsenal,Stoke,A,0,1,0,1,3.0,4.000000,3.000000,4.000000,3.000000,8
2,1718,2017-08-27,Arsenal,Liverpool,A,0,4,0,2,3.0,2.000000,2.000000,2.000000,2.000000,8
3,1718,2017-09-09,Arsenal,Bournemouth,H,3,0,3,3,3.0,1.333333,2.666667,1.333333,2.666667,13
4,1718,2017-09-17,Arsenal,Chelsea,A,0,0,1,4,6.0,1.750000,2.000000,1.750000,2.000000,8
5,1718,2017-09-25,Arsenal,West Brom,H,2,0,3,5,7.0,1.400000,1.600000,1.400000,1.600000,8
6,1718,2017-10-01,Arsenal,Brighton,H,2,0,3,5,7.0,1.000000,1.000000,1.500000,1.333333,6
7,1718,2017-10-14,Arsenal,Watford,A,1,2,0,5,10.0,1.400000,0.800000,1.571429,1.142857,13
8,1718,2017-10-22,Arsenal,Everton,A,5,2,3,5,10.0,1.600000,0.400000,1.500000,1.250000,8
9,1718,2017-10-28,Arsenal,Swansea,H,2,1,3,5,10.0,2.000000,0.800000,1.888889,1.333333,6


In [22]:
con.sql("""
    SELECT (SELECT COUNT(*) FROM team_form) AS form_rows,
           (SELECT COUNT(*) FROM team_matches) AS team_rows
""").df()

,form_rows,team_rows
0,7040,7040


In [23]:
con.sql("SELECT COUNT(*) FROM team_form WHERE prior_matches = 0").df()

,count_star()
0,34


In [24]:
df = con.sql("SELECT * FROM team_form WHERE Team = 'Arsenal' ORDER BY Date").df()

manual = df["Points"].shift(1).rolling(5).sum()    # shift(1) excludes the current match
full = df["prior_matches"] == 5

(df.loc[full, "points_last5"] - manual[full]).abs().max()    # should be 0

np.float64(0.0)

In [25]:
con.sql("""
    COPY (SELECT * FROM team_form ORDER BY Date, Team)
    TO '../data/processed/team_form.csv' (HEADER, DELIMITER ',')
""")

## Summary: what this notebook built

**Views created in DuckDB**
- `matches`: the cleaned match data, read directly from `data/processed/matches.csv`
- `team_matches`: one row per team per match, so goals and points sit in one column
- `team_form`: rolling features for each team before each match (points, goals scored and conceded over the last 5 and 10 matches, days of rest)

**Checks that passed**
- SQL results for matches per season, home win rate and goals per match agree with the pandas results
- `team_matches` has exactly twice as many rows as `matches`, with 38 matches per team in each completed season
- The 2025/26 league table built from `team_matches` matches the real final standings (champions [team], [X] points)
- `team_form` has the same number of rows as `team_matches`, and each team's first match has no history (34 rows, one per distinct team)
- Recomputing `points_last5` in pandas gives the same values as the SQL window (difference of 0)

**Key idea: avoiding data leakage**
Every rolling window ends at `1 PRECEDING`, so a match's features only use games played before it. Using `CURRENT ROW` would put the match's own result into its features, which would inflate test results and fail on real future matches.

**Things to handle in the modeling stage**
- The first matches of each team (and promoted teams) have missing or thin history, so they need a rule such as dropping rows with fewer than 5 prior matches
- Form carries over between seasons, so a team's late-season matches feed its early-season features
- Shots, fouls and cards from the current match must never be used as features, only rolling averages of previous matches

## Next: Stage 3, modeling
1. Build a one-row-per-match training table (home features and away features side by side, with `FTHG` and `FTAG` as targets)
2. Baselines: league-average goals for every match, then simple team ratings
3. Poisson regression and LightGBM to predict expected goals for each team
4. Convert expected goals into scoreline and win/draw/loss probabilities, then evaluate on newer seasons against the baseline and bookmaker odds